# Аналитика логов сайта

Ноутбук читает файл `debug.log`, разбирает JSON-логи приложения и считает простые метрики для отчёта.

Файл логов должен лежать рядом с проектом:

```text
debug.log
```

## 1. Импорт библиотек

In [ ]:
import json
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_colwidth', 120)

## 2. Чтение `debug.log`

In [ ]:
log_file = Path('debug.log')

if not log_file.exists():
    raise FileNotFoundError('Файл debug.log не найден. Проверьте путь или загрузите файл в Colab.')

records = []

with log_file.open('r', encoding='utf-8') as file:
    for line_number, line in enumerate(file, start=1):
        line = line.strip()
        if not line:
            continue
        try:
            records.append(json.loads(line))
        except json.JSONDecodeError:
            print(f'Пропущена строка {line_number}: некорректный JSON')

df = pd.DataFrame(records)
df.head()

## 3. Подготовка таблицы

In [ ]:
df['datetime'] = pd.to_datetime(df['datetime'], errors='coerce')

df['count'] = df['data'].apply(lambda x: x.get('count') if isinstance(x, dict) else None)
df['decision'] = df['data'].apply(lambda x: x.get('decision') if isinstance(x, dict) else None)
df['error'] = df['data'].apply(lambda x: x.get('error') if isinstance(x, dict) else None)

df[['datetime', 'hypothesisId', 'message', 'count', 'decision', 'error']].head(10)

## 4. Основные метрики

- **Количество посещений** считается по событиям `Counter incremented`.
- **Решения приложения** считаются по событиям `Decision selected`.
- **Ошибки** определяются по наличию поля `error`.

In [ ]:
visits = df[df['message'] == 'Counter incremented'].copy()
decisions = df[df['message'] == 'Decision selected'].copy()
errors = df[df['error'].notna()].copy()

metrics = {
    'Всего записей в логах': len(df),
    'Количество посещений сайта': len(visits),
    'Текущее значение счетчика Redis': int(visits['count'].max()) if not visits.empty else 0,
    'Количество решений приложения': len(decisions),
    'Количество ошибок': len(errors),
    'Первое событие': df['datetime'].min(),
    'Последнее событие': df['datetime'].max(),
}

metrics_df = pd.DataFrame(metrics.items(), columns=['Метрика', 'Значение'])
metrics_df

## 5. Распределение сценариев

Показывает, какие варианты бизнес-логики срабатывали чаще всего.

In [ ]:
decision_stats = (
    decisions['decision']
    .value_counts()
    .rename_axis('Решение')
    .reset_index(name='Количество')
)

if not decision_stats.empty:
    decision_stats['Доля, %'] = (decision_stats['Количество'] / decision_stats['Количество'].sum() * 100).round(2)

decision_stats

In [ ]:
if not decision_stats.empty:
    plt.figure(figsize=(10, 5))
    plt.bar(decision_stats['Решение'], decision_stats['Количество'])
    plt.title('Распределение решений приложения')
    plt.xlabel('Решение')
    plt.ylabel('Количество')
    plt.xticks(rotation=30, ha='right')
    plt.tight_layout()
    plt.show()
else:
    print('Нет данных для графика.')

## 6. Посещения по времени

График помогает увидеть активность сайта по секундам.

In [ ]:
if not visits.empty:
    visits_by_second = visits.set_index('datetime').resample('1s').size().reset_index(name='Посещения')
    visits_by_second
else:
    visits_by_second = pd.DataFrame(columns=['datetime', 'Посещения'])
    print('Посещения не найдены.')

In [ ]:
if not visits_by_second.empty:
    plt.figure(figsize=(10, 5))
    plt.plot(visits_by_second['datetime'], visits_by_second['Посещения'], marker='o')
    plt.title('Посещения сайта по секундам')
    plt.xlabel('Время')
    plt.ylabel('Количество посещений')
    plt.xticks(rotation=30, ha='right')
    plt.tight_layout()
    plt.show()
else:
    print('Нет данных для графика.')

## 7. Проверка ошибок

In [ ]:
if errors.empty:
    print('Ошибки в логах не обнаружены.')
else:
    errors[['datetime', 'hypothesisId', 'message', 'error']]

## 8. Итог для отчёта

In [ ]:
total_visits = len(visits)
max_count = int(visits['count'].max()) if not visits.empty else 0
error_count = len(errors)

top_decision = 'нет данных'
if not decision_stats.empty:
    top_decision = decision_stats.iloc[0]['Решение']

print('Итог анализа:')
print(f'- Зафиксировано посещений сайта: {total_visits}')
print(f'- Текущее значение счетчика Redis: {max_count}')
print(f'- Самый частый сценарий приложения: {top_decision}')
print(f'- Количество ошибок: {error_count}')

if error_count == 0:
    print('- Приложение работает стабильно, критические ошибки не обнаружены.')
else:
    print('- В логах есть ошибки, требуется проверить Redis и контейнеры.')